# Fase 1 – Analisi e validazione del CSV per AI_microtradingAUX

Questo notebook raccoglie la spiegazione e il codice necessari per completare la **Fase 1** del progetto `AI_microtradingAUX`: analisi e validazione del CSV M1 con pattern `Date;Open;High;Low;Close;Volume`.

Obiettivi principali:

- Caricare il file con separatore `;` (non `;;`).
- Convertire la colonna `Date` in `DatetimeIndex` ordinato cronologicamente.
- Verificare duplicati di timestamp e rimuoverli secondo la policy configurata.
- Trovare intervalli mancanti rispetto alla frequenza attesa (es. 1 minuto).
- Controllare la coerenza OHLC (High/Low vs Open/Close).
- Analizzare il volume (zero/negative volume, media).
- Identificare weekend e chiusure prolungate.
- **Non** assumere automaticamente il fuso orario del broker, ma documentarne l’analisi.


## Contesto del progetto

Il progetto `AI_microtradingAUX` utilizza un file CSV M1 (es. `data/raw/XAU_1m_data.csv`) e un file di configurazione `config.yaml` per definire:

- Percorso del CSV (`raw_path`).
- Separatore (`separator`, tipicamente `;`).
- Colonna timestamp (`timestamp_column`, es. `Date`).
- Formato della data (`timestamp_format`, es. `%Y.%m.%d %H:%M`).
- Policy sui duplicati (`duplicate_policy`).
- Frequenza attesa (`expected_frequency`, es. `1min`).

La logica di caricamento e validazione base dei dati è già implementata in `src/gold_rl/data/loader.py` (funzioni `load_market_data`, `handle_duplicates`, `validate_ohlcv`, `get_dataset_summary`).

## Estensioni a `loader.py` per gap, calendario e data_quality

In questa sezione definiamo tre funzioni da aggiungere a `src/gold_rl/data/loader.py`:

1. `get_gaps_table`: costruisce una tabella dettagliata dei gap temporali rispetto alla frequenza attesa.
2. `analyze_calendar`: analizza il calendario del dataset (weekend, settimane con dati, chiusure prolungate) senza assegnare automaticamente una timezone.
3. `build_data_quality_report`: compone un report strutturato compatibile con `reports/data_quality.json`, riassumendo schema, integrità, volume, calendario e note sul fuso orario.


In [ ]:
from typing import Any
import pandas as pd


def get_gaps_table(
    dataframe: pd.DataFrame,
    expected_frequency: str = "1min",
) -> pd.DataFrame:
    """
    Costruisce una tabella con i gap temporali rilevati.

    Ogni riga rappresenta un intervallo in cui mancano una o più barre
    rispetto alla frequenza attesa (es. 1 minuto).
    """
    if dataframe.empty:
        raise ValueError(
            "Impossibile analizzare i gap di un dataset vuoto."
        )

    if not isinstance(dataframe.index, pd.DatetimeIndex):
        raise TypeError(
            "L'indice del DataFrame deve essere un DatetimeIndex."
        )

    expected_delta = pd.to_timedelta(expected_frequency)

    # Differenze tra timestamp consecutivi.
    time_differences = (
        dataframe.index
        .to_series()
        .diff()
    )

    gap_mask = time_differences > expected_delta
    gaps = time_differences.loc[gap_mask]

    if gaps.empty:
        return pd.DataFrame(
            columns=["gap_start", "gap_end", "gap_duration", "missing_bars"]
        )

    rows: list[dict[str, Any]] = []

    # Per ogni gap individuiamo inizio, fine e barre mancanti.
    for gap_timestamp, gap_delta in gaps.items():
        # gap_timestamp è il timestamp di arrivo del gap.
        # Il timestamp precedente è l'ultima barra prima del vuoto.
        previous_timestamp = dataframe.index[
            dataframe.index.get_loc(gap_timestamp) - 1
        ]

        missing_bars = int(
            (gap_delta / expected_delta) - 1
        )

        rows.append(
            {
                "gap_start": previous_timestamp,
                "gap_end": gap_timestamp,
                "gap_duration": gap_delta,
                "missing_bars": missing_bars,
            }
        )

    gaps_table = pd.DataFrame(rows)

    return gaps_table


def analyze_calendar(
    dataframe: pd.DataFrame,
    long_closure_threshold: str = "2D",
) -> dict[str, Any]:
    """
    Analizza il calendario del dataset per individuare weekend
    e chiusure prolungate (es. weekend FX, festività).

    Non assegna automaticamente il fuso orario: fornisce solo
    indicatori diagnostici.
    """
    if dataframe.empty:
        raise ValueError(
            "Impossibile analizzare il calendario di un dataset vuoto."
        )

    if not isinstance(dataframe.index, pd.DatetimeIndex):
        raise TypeError(
            "L'indice del DataFrame deve essere un DatetimeIndex."
        )

    # Conteggio barre per giorno della settimana (0=lunedì, 6=domenica).
    day_counts = (
        dataframe.index.dayofweek
        .value_counts()
        .sort_index()
    )

    weekend_bars = int(
        day_counts.get(5, 0) + day_counts.get(6, 0)
    )

    # Numero di settimane con almeno una barra.
    weeks = dataframe.index.to_period("W")
    weeks_with_data = int(
        weeks.value_counts().shape[0]
    )

    # Giorni di trading distinti per settimana (media).
    trading_days_per_week: list[int] = []
    for week, group in dataframe.groupby(weeks):
        unique_days = group.index.dayofweek.unique()
        trading_days_per_week.append(len(unique_days))

    average_trading_days_per_week = float(
        pd.Series(trading_days_per_week).mean()
    )

    # Chiusure prolungate: gap più lunghi di una soglia (es. 2 giorni).
    threshold = pd.to_timedelta(long_closure_threshold)
    expected_delta = pd.to_timedelta("1min")

    time_differences = (
        dataframe.index
        .to_series()
        .diff()
    )
    long_gap_mask = time_differences > threshold
    long_gaps = time_differences.loc[long_gap_mask]

    long_closures: list[dict[str, Any]] = []

    for gap_timestamp, gap_delta in long_gaps.items():
        previous_timestamp = dataframe.index[
            dataframe.index.get_loc(gap_timestamp) - 1
        ]

        missing_bars = int(
            (gap_delta / expected_delta) - 1
        )

        long_closures.append(
            {
                "start": previous_timestamp,
                "end": gap_timestamp,
                "duration": gap_delta,
                "estimated_missing_bars": missing_bars,
            }
        )

    calendar_summary: dict[str, Any] = {
        "bars_per_day_of_week": {
            int(day): int(count)
            for day, count in day_counts.items()
        },
        "weekend_bars": weekend_bars,
        "weeks_with_data": weeks_with_data,
        "average_trading_days_per_week": average_trading_days_per_week,
        "long_closures": long_closures,
        # Nessun fuso orario assegnato automaticamente.
        "timezone": None,
        "timezone_note": (
            "Il fuso orario del broker deve essere determinato da "
            "metadati esterni (es. documentazione del broker, "
            "label EET/EEST, impostazioni MT4/MT5) e NON "
            "assunto automaticamente."
        ),
    }

    return calendar_summary


def build_data_quality_report(
    dataframe: pd.DataFrame,
    expected_frequency: str = "1min",
    separator: str = ";",
    timestamp_column: str = "Date",
    timestamp_format: str = "%Y.%m.%d %H:%M",
) -> dict[str, Any]:
    """
    Costruisce un dizionario compatibile con reports/data_quality.json
    che riassume qualità, calendario e volume del dataset M1.
    """
    from gold_rl.data.loader import get_dataset_summary

    summary = get_dataset_summary(
        dataframe=dataframe,
        expected_frequency=expected_frequency,
    )

    calendar = analyze_calendar(dataframe)

    report: dict[str, Any] = {
        "schema": {
            "columns": list(dataframe.columns),
            "separator": separator,
            "timestamp_column": timestamp_column,
            "timestamp_format": timestamp_format,
            # Nessuna timezone assegnata in automatico.
            "timezone": calendar["timezone"],
        },
        "integrity": {
            "rows": summary["rows"],
            "duplicate_timestamps": summary["duplicate_timestamps"],
            "missing_values": summary["missing_values"],
            "detected_gaps": summary["detected_gaps"],
            "estimated_missing_bars": summary["estimated_missing_bars"],
            "largest_gap": summary["largest_gap"],
            "median_interval": summary["median_interval"],
        },
        "volume": {
            "zero_volume_bars": summary["zero_volume_bars"],
            "negative_volume_bars": summary["negative_volume_bars"],
            "average_volume": summary["average_volume"],
        },
        "price": {
            "minimum_close": summary["minimum_close"],
            "maximum_close": summary["maximum_close"],
        },
        "calendar": calendar,
        "notes": {
            "timezone_note": calendar["timezone_note"],
            "frequency_assumption": (
                "Analisi basata su expected_frequency="
                f"{expected_frequency}. I minuti senza "
                "transazioni possono generare gap intraday "
                "legittimi su timeframe M1."
            ),
        },
    }

    return report


## Script `scripts/build_data_quality.py`

Ora definiamo uno script che usa le funzioni di caricamento dati e le nuove funzioni di analisi per generare automaticamente i file:

- `reports/data_quality.json`: report strutturato di qualità e calendario.
- `reports/gaps.csv`: tabella dei gap temporali.
- `reports/data_summary.md`: riepilogo human‑readable dei principali indicatori.


In [ ]:
from pathlib import Path
import json
import sys

import yaml
import pandas as pd

PROJECT_ROOT = Path(__file__).resolve().parents[1]
SRC_DIRECTORY = PROJECT_ROOT / "src"
REPORTS_DIRECTORY = PROJECT_ROOT / "reports"

# Permette a Python di trovare il package src/gold_rl.
sys.path.insert(0, str(SRC_DIRECTORY))

REPORTS_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)

from gold_rl.data.loader import (
    build_data_quality_report,
    get_dataset_summary,
    get_gaps_table,
    load_market_data,
)


def load_config() -> dict:
    """Carica la configurazione principale del progetto."""
    config_path = PROJECT_ROOT / "config.yaml"

    if not config_path.exists():
        raise FileNotFoundError(
            f"Configurazione non trovata: {config_path}"
        )

    with config_path.open(
        mode="r",
        encoding="utf-8",
    ) as config_file:
        config = yaml.safe_load(config_file)

    if not isinstance(config, dict):
        raise ValueError(
            "Il file config.yaml è vuoto oppure non è valido."
        )

    if "data" not in config:
        raise KeyError(
            "Nel file config.yaml manca la sezione 'data'."
        )

    return config


def save_data_quality_json(report: dict) -> Path:
    """Salva reports/data_quality.json."""
    output_path = REPORTS_DIRECTORY / "data_quality.json"

    with output_path.open(
        mode="w",
        encoding="utf-8",
    ) as output_file:
        json.dump(
            report,
            output_file,
            indent=4,
            ensure_ascii=False,
            default=str,
        )

    return output_path


def save_gaps_csv(gaps_table: pd.DataFrame) -> Path:
    """Salva reports/gaps.csv con la tabella dei gap."""
    output_path = REPORTS_DIRECTORY / "gaps.csv"

    gaps_table.to_csv(
        output_path,
        index=False,
        encoding="utf-8",
    )

    return output_path


def save_data_summary_md(
    summary: dict,
    calendar_report: dict,
) -> Path:
    """
    Salva un riepilogo human-readable in reports/data_summary.md.

    Riassume durata, volume, gap e calendario (weekend/chiusure).
    """
    output_path = REPORTS_DIRECTORY / "data_summary.md"

    lines: list[str] = []

    lines.append("# Data Summary\n")
    lines.append("## Overview\n")
    lines.append(
        f"- Rows: {summary['rows']}\n"
        f"- Start: {summary['start']}\n"
        f"- End: {summary['end']}\n"
        f"- Duration (days): {summary['duration_days']}\n"
        f"- Memory (MB): {summary['memory_mb']}\n"
    )

    lines.append("## Integrity\n")
    lines.append(
        f"- Duplicate timestamps: {summary['duplicate_timestamps']}\n"
        f"- Missing values: {summary['missing_values']}\n"
        f"- Detected gaps: {summary['detected_gaps']}\n"
        f"- Estimated missing bars: {summary['estimated_missing_bars']}\n"
        f"- Largest gap: {summary['largest_gap']}\n"
        f"- Median interval: {summary['median_interval']}\n"
    )

    lines.append("## Volume\n")
    lines.append(
        f"- Zero-volume bars: {summary['zero_volume_bars']}\n"
        f"- Negative-volume bars: {summary['negative_volume_bars']}\n"
        f"- Average volume: {summary['average_volume']}\n"
    )

    lines.append("## Prices\n")
    lines.append(
        f"- Minimum close: {summary['minimum_close']}\n"
        f"- Maximum close: {summary['maximum_close']}\n"
    )

    lines.append("## Calendar\n")
    lines.append(
        f"- Weekend bars: {calendar_report['weekend_bars']}\n"
        f"- Weeks with data: {calendar_report['weeks_with_data']}\n"
        f"- Average trading days per week: {calendar_report['average_trading_days_per_week']}\n"
        f"- Bars per day of week: {calendar_report['bars_per_day_of_week']}\n"
        f"- Long closures (>=2D): {calendar_report['long_closures']}\n"
    )

    lines.append("## Timezone\n")
    lines.append(
        f"- Timezone: {calendar_report['timezone']}\n"
        f"- Note: {calendar_report['timezone_note']}\n"
    )

    with output_path.open(
        mode="w",
        encoding="utf-8",
    ) as output_file:
        output_file.write("\n".join(lines))

    return output_path


def main() -> None:
    config = load_config()
    data_config = config["data"]

    csv_path = PROJECT_ROOT / data_config["raw_path"]

    print(f"Caricamento dataset: {csv_path.resolve()}")

    dataframe = load_market_data(
        file_path=csv_path,
        separator=data_config["separator"],
        timestamp_column=data_config["timestamp_column"],
        timestamp_format=data_config["timestamp_format"],
        duplicate_policy=data_config["duplicate_policy"],
    )

    summary = get_dataset_summary(
        dataframe=dataframe,
        expected_frequency=data_config["expected_frequency"],
    )

    gaps_table = get_gaps_table(
        dataframe=dataframe,
        expected_frequency=data_config["expected_frequency"],
    )

    data_quality_report = build_data_quality_report(
        dataframe=dataframe,
        expected_frequency=data_config["expected_frequency"],
        separator=data_config["separator"],
        timestamp_column=data_config["timestamp_column"],
        timestamp_format=data_config["timestamp_format"],
    )

    calendar_report = data_quality_report["calendar"]

    quality_path = save_data_quality_json(data_quality_report)
    gaps_path = save_gaps_csv(gaps_table)
    summary_md_path = save_data_summary_md(summary, calendar_report)

    print("\nReport generati:")
    print(f"- data_quality.json: {quality_path.resolve()}")
    print(f"- gaps.csv: {gaps_path.resolve()}")
    print(f"- data_summary.md: {summary_md_path.resolve()}")


if __name__ == "__main__":
    main()


## Come usare il notebook e integrare il codice nel progetto

1. Copia le funzioni `get_gaps_table`, `analyze_calendar`, `build_data_quality_report` dentro `src/gold_rl/data/loader.py` (alla fine del file, dopo `get_dataset_summary`).
2. Crea il file `scripts/build_data_quality.py` usando il codice dello script.
3. Assicurati che `config.yaml` abbia una sezione `data` con i campi `raw_path`, `separator`, `timestamp_column`, `timestamp_format`, `duplicate_policy`, `expected_frequency`.
4. Posiziona il CSV grande in `data/raw/XAU_1m_data.csv` (o nel path indicato da `raw_path`).
5. Esegui `python scripts/build_data_quality.py` dalla root del progetto per generare automaticamente `reports/data_quality.json`, `reports/gaps.csv`, `reports/data_summary.md`.

In questo modo completi la Fase 1 rispettando il vincolo di non assumere automaticamente il fuso orario del broker, ma documentandolo nella sezione `calendar/timezone` del report di qualità.
